In [ ]:
"""
Object Detection using Faster R-CNN ResNet50 FPN
Detects 80 COCO object classes in video frames using PyTorch + OpenCV

Usage:
    python object_detection.py input_video.mp4
    python object_detection.py  (uses default 'input_video.mp4')

Author: Bhanu Priya Palivela
GitHub: github.com/priyapalivela
"""

import sys
import cv2
import torch
from torchvision import models, transforms
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights

# ── COCO class labels (80 classes) ───────────────────────────────────────────
COCO_LABELS = [
    '__background__', 'person', 'bicycle', 'car', 'motorcycle',
    'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light',
    'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird',
    'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear',
    'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie',
    'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball',
    'kite', 'baseball bat', 'baseball glove', 'skateboard',
    'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup',
    'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple',
    'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza',
    'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed',
    'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote',
    'keyboard', 'cell phone', 'microwave', 'oven', 'toaster',
    'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors',
    'teddy bear', 'hair drier', 'toothbrush'
]

# ── Config ────────────────────────────────────────────────────────────────────
CONFIDENCE_THRESHOLD = 0.5
video_path  = sys.argv[1] if len(sys.argv) > 1 else 'input_video.mp4'
output_path = 'output_video.mp4'

# ── Load pretrained Faster R-CNN model ───────────────────────────────────────
print("[INFO] Loading Faster R-CNN ResNet50 FPN model...")
model = models.detection.fasterrcnn_resnet50_fpn(
    weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT
)
model.eval()
print("[INFO] Model loaded successfully.")

# ── Transformation ────────────────────────────────────────────────────────────
transform = transforms.Compose([transforms.ToTensor()])

# ── Open video ────────────────────────────────────────────────────────────────
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise FileNotFoundError(f"[ERROR] Could not open video: {video_path}")

frame_width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps          = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"[INFO] Video: {video_path} | {frame_width}x{frame_height} | {fps:.1f} FPS | {total_frames} frames")

# ── Output video writer ───────────────────────────────────────────────────────
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out    = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))

# ── Inference loop ────────────────────────────────────────────────────────────
frame_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1
    if frame_count % 10 == 0:
        print(f"[INFO] Processing frame {frame_count}/{total_frames}...")

    # Preprocess frame: BGR → RGB → tensor
    rgb_frame    = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    input_tensor = transform(rgb_frame).unsqueeze(0)

    # Run inference
    with torch.no_grad():
        detections = model(input_tensor)[0]

    # Extract detections
    boxes  = detections['boxes'].cpu().numpy()
    scores = detections['scores'].cpu().numpy()
    labels = detections['labels'].cpu().numpy()

    # Filter by confidence threshold
    indices = [i for i, score in enumerate(scores) if score > CONFIDENCE_THRESHOLD]

    # Draw bounding boxes
    for i in indices:
        box        = boxes[i]
        x1, y1, x2, y2 = box.astype(int)
        class_name = COCO_LABELS[labels[i]] if labels[i] < len(COCO_LABELS) else f"class_{labels[i]}"
        label_text = f'{class_name}: {scores[i]:.2f}'

        # Draw rectangle and label
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(
            frame, label_text,
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2
        )

    # Write annotated frame
    out.write(frame)

    # Display (press Q to quit)
    cv2.imshow('Object Detection — Faster R-CNN', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        print("[INFO] Stopped by user.")
        break

# ── Cleanup ───────────────────────────────────────────────────────────────────
cap.release()
out.release()
cv2.destroyAllWindows()

print(f"[INFO] Done! Processed {frame_count} frames.")
print(f"[INFO] Output saved to: {output_path}")